# Auditoría de Ground Truth, Análisis de Errores y Mitigación del Contagio Sintáctico en FinBERT

**Autor / Proyecto:** TFM Data Science — NLP Financiero y Predicción de Sentimiento Bancario  
**Objetivo:** Este notebook formaliza el proceso de auditoría *Human-in-the-Loop* sobre una muestra de control de 30 noticias reales de la capa Silver (`scratch/finbert_pilot_v2_revision_manual.csv`), diagnosticando un sesgo crítico en modelos preentrenados (**Contagio Sintáctico**) y validando experimentalmente su solución arquitectónica antes de modificar el código de producción.

In [1]:
# 0. Configuración del entorno y dependencias
import sys, os, re, time
from pathlib import Path

curr = Path.cwd()
PROJ_ROOT = next((p for p in [curr, *curr.parents] if (p / "pyproject.toml").exists()), curr)
for p in (PROJ_ROOT / "src", PROJ_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import spacy
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from shared.tickets import ALIAS_BANCOS

nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
finbert = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").eval()

LBL = {v.lower(): k for k, v in finbert.config.id2label.items()}
I_POS, I_NEG, I_NEU = LBL["positive"], LBL["negative"], LBL["neutral"]
print("[OK] Entorno listo. FinBERT cargado.")

c:\Users\felip\OneDrive\Felipe\Master\TFM\TFM-Master\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 275.26it/s]

[OK] Entorno listo. FinBERT cargado.


## 1. Carga del Dataset de Control y Ground Truth Humano

Cargamos la muestra de 30 noticias auditadas manualmente (`scratch/finbert_pilot_v2_revision_manual.csv`). La columna `etiqueta_manual` representa el criterio financiero experto:
- `+1` (**Positivo**): Beneficios que baten previsiones, ingresos récord o subidas bursátiles.
- `0` (**Neutral**): Calendarios de resultados, mandatos rutinarios M&A, análisis de múltiplos P/B o notas macro.
- `-1` (**Negativo**): Recortes de plantilla, pérdidas de colocación de deuda, provisiones por mora o sanciones regulatorias.

In [2]:
# Cargar dataset de auditoría
csv_path = PROJ_ROOT / "scratch" / "finbert_pilot_v2_revision_manual.csv"
df_audit = pd.read_csv(csv_path)

print(f"Muestra de auditoría cargada: {len(df_audit)} artículos.")
dist_gt = df_audit["etiqueta_manual"].value_counts().rename({1: "Positivo (+1)", 0: "Neutral (0)", -1: "Negativo (-1)"})
display(pd.DataFrame({"Conteo": dist_gt, "Porcentaje (%)": (100 * dist_gt / len(df_audit)).round(1)}))
display(df_audit[["ticker", "titular", "etiqueta_manual", "R1_sent", "S1_sent", "S1B_sent"]].head(5))

Muestra de auditoría cargada: 30 artículos.


,Conteo,Porcentaje (%)
etiqueta_manual,,
Neutral (0),16,53.3
Positivo (+1),8,26.7
Negativo (-1),6,20.0


,ticker,titular,etiqueta_manual,R1_sent,S1_sent,S1B_sent
0,C,"Zacks Earnings Trends Highlights: JPMorgan, Ci...",0,-0.91,-0.95,-0.94
1,C,"Financial Sector Update for 03/10/2016: BKCC,A...",0,0.12,0.12,0.12
2,MS,French retailer Carrefour to launch $1.9 bln s...,0,-0.00,-0.00,-0.00
3,WFC,"Citigroup (C) Realigns Corporate Structure, Tr...",-1,-0.96,-0.95,-0.96
4,GS,Hg hires Goldman Sachs to prepare sale of Germ...,0,0.26,0.13,0.13


## 2. Benchmark Formal: Métricas de Machine Learning frente al Criterio Humano

Evaluamos cuantitativamente las cuatro estrategias del Piloto v2 frente al *Ground Truth* humano discretizando la polaridad continua con el umbral estándar $|S| > 0.15$.

In [3]:
def discretizar(s, umbral=0.15):
    return s.apply(lambda x: 1 if x > umbral else (-1 if x < -umbral else 0))

y_true = df_audit["etiqueta_manual"]
estrategias = ["R1", "S1", "S2", "S1B"]

filas_kpis = []
for est in estrategias:
    y_pred = discretizar(df_audit[f"{est}_sent"])
    acc = (y_true == y_pred).mean()
    rec_pos = (y_pred[y_true == 1] == 1).mean() if (y_true == 1).sum() > 0 else 0
    rec_neu = (y_pred[y_true == 0] == 0).mean() if (y_true == 0).sum() > 0 else 0
    rec_neg = (y_pred[y_true == -1] == -1).mean() if (y_true == -1).sum() > 0 else 0
    inv = (((y_true == 1) & (y_pred == -1)) | ((y_true == -1) & (y_pred == 1))).sum()
    filas_kpis.append({
        "Estrategia": est,
        "Accuracy Global": acc,
        "Recall Negativo (-1)": rec_neg,
        "Recall Neutral (0)": rec_neu,
        "Recall Positivo (+1)": rec_pos,
        "Inversiones Graves": inv
    })

df_kpis = pd.DataFrame(filas_kpis).round(3)
display(df_kpis)

,Estrategia,Accuracy Global,Recall Negativo (-1),Recall Neutral (0),Recall Positivo (+1),Inversiones Graves
0,R1,0.433,0.667,0.500,0.125,6
1,S1,0.433,0.667,0.562,0.000,6
2,S2,0.433,0.667,0.375,0.375,4
3,S1B,0.500,0.833,0.562,0.125,6


## 3. Diagnóstico del Fenómeno: El "Contagio Sintáctico" en FinBERT

Al auditar las discrepancias, se observa que el **Recall Positivo es anormalmente bajo (12.5%)** en las variantes de ventana $\pm 1$.

### ¿Por qué ocurre esto?
FinBERT posee cabezas de atención con una fuerte **aversión a la pérdida**. Cuando en la frase adyacente ($i-1$ o $i+1$) aparece léxico de caídas o crisis de activos ajenos (*"First Republic se desploma un 41%"*, *"Techs caen"* o *"Pearson pierde 29%"*), la atención del Transformer sobrepondera esas palabras negativas y **contamina la señal del banco objetivo**, asignándole un sentimiento bajista a pesar de que el banco batió beneficios.

In [4]:
# Inspección de los 3 casos paradigmáticos de falso negativo por contagio
casos_contagio = [9, 12, 27]
for idx in casos_contagio:
    row = df_audit.iloc[idx]
    print(f"\n{'='*80}")
    print(f"[FILA {idx:02d}] Ticker Objetivo: [{row['ticker']}] | Humano: +1 (Positivo) | FinBERT S1B: {row['S1B_sent']:.2f} (FALSO NEGATIVO)")
    print(f"Titular: {row['titular']}")
    print(f"Texto que leyó FinBERT (Ventana ±1):\n{row['S1B_texto'][:400]}...")


[FILA 09] Ticker Objetivo: [MS] | Humano: +1 (Positivo) | FinBERT S1B: -0.88 (FALSO NEGATIVO)
Titular: US STOCKS-S&P, Dow gain as bank fears ease ahead of Fed meet
Texto que leyó FinBERT (Ventana ±1):
US STOCKS-S&P, Dow gain as bank fears ease ahead of Fed meet [SEP] UBS reversed premarket declines to jump 2.9%. Big U.S. banks such as JPMorgan Chase & Co JPM.N, Citigroup C.N and Morgan Stanley MS.N rose between 1.1% and 1.9%. Regional bank First Republic Bank FRC.N slid nearly 41% following a downgrade by S&P Global and a report of more fundraising that fanned worries about the bank's liquidity...

[FILA 12] Ticker Objetivo: [GS] | Humano: +1 (Positivo) | FinBERT S1B: -0.96 (FALSO NEGATIVO)
Titular: Mid-Morning Market Update: Markets Mixed; Goldman Sachs Profit Beats Views
Texto que leyó FinBERT (Ventana ±1):
Mid-Morning Market Update: Markets Mixed; Goldman Sachs Profit Beats Views [SEP] Meanwhile, top losers in the sector included Pearson PLC (ADR) (NYSE: PSO ), down 29 percent, and

## 4. Experimento de Mitigación: Extracción Estricta (Ventana 0 / Foco de Entidad)

**Hipótesis:** Si eliminamos las oraciones adyacentes que no mencionan al banco objetivo y evaluamos exclusivamente `Titular [SEP] Frase_Exacta_del_Banco` (**Ventana 0 / `S0`**), aislamos al modelo de la contaminación de competidores y el Recall Positivo debería recuperarse drásticamente sin perder la precisión en negativos.

In [5]:
@torch.inference_mode()
def score_text(txt):
    inputs = tokenizer(txt, return_tensors="pt", truncation=True, max_length=512)
    probs = torch.softmax(finbert(**inputs).logits, dim=-1)[0]
    return (probs[I_POS] - probs[I_NEG]).item()

def extraer_ventana_0(titular, s1b_texto, ticker):
    aliases = ALIAS_BANCOS.get(ticker, [ticker.lower()])
    pat = re.compile(r"\b(" + "|".join(re.escape(a) for a in aliases) + r")\b", re.IGNORECASE)
    partes = s1b_texto.split(" [SEP] ")
    cuerpo = partes[1] if len(partes) > 1 else s1b_texto
    sents = [s.text.strip() for s in nlp(cuerpo).sents if s.text.strip()]
    hits = [s for s in sents if pat.search(s) or pat.search(re.sub(r"\s+", " ", s.replace(".", "").replace("-", " ")))]
    if not hits:
        return f"{titular.strip()} [SEP] {sents[0] if sents else ''}"
    return f"{titular.strip()} [SEP] " + " ".join(hits)

# Ejecución del experimento S0
scores_s0 = [score_text(extraer_ventana_0(r["titular"], r["S1B_texto"], r["ticker"])) for _, r in df_audit.iterrows()]
df_audit["S0_sent"] = scores_s0

y_pred_s0 = discretizar(df_audit["S0_sent"])
y_pred_s1b = discretizar(df_audit["S1B_sent"])

comp_data = [
    {
        "Estrategia": "S1B (Ventana ±1 + Poda)",
        "Accuracy Global": (y_true == y_pred_s1b).mean(),
        "Recall Negativo (-1)": (y_pred_s1b[y_true == -1] == -1).mean(),
        "Recall Neutral (0)": (y_pred_s1b[y_true == 0] == 0).mean(),
        "Recall Positivo (+1)": (y_pred_s1b[y_true == 1] == 1).mean(),
        "Inversiones Graves": (((y_true == 1) & (y_pred_s1b == -1)) | ((y_true == -1) & (y_pred_s1b == 1))).sum()
    },
    {
        "Estrategia": "S0 (Foco Estricto / Ventana 0)",
        "Accuracy Global": (y_true == y_pred_s0).mean(),
        "Recall Negativo (-1)": (y_pred_s0[y_true == -1] == -1).mean(),
        "Recall Neutral (0)": (y_pred_s0[y_true == 0] == 0).mean(),
        "Recall Positivo (+1)": (y_pred_s0[y_true == 1] == 1).mean(),
        "Inversiones Graves": (((y_true == 1) & (y_pred_s0 == -1)) | ((y_true == -1) & (y_pred_s0 == 1))).sum()
    }
]
display(pd.DataFrame(comp_data).round(3))

,Estrategia,Accuracy Global,Recall Negativo (-1),Recall Neutral (0),Recall Positivo (+1),Inversiones Graves
0,S1B (Ventana ±1 + Poda),0.500,0.833,0.562,0.125,6
1,S0 (Foco Estricto / Ventana 0),0.667,0.833,0.562,0.750,1


## 5. Comparativa Técnica y Propuestas Arquitectónicas para Discusión en Equipo

A la vista de los resultados empíricos, planteamos **tres soluciones concretas** para debatir en equipo antes de tocar código de producción:

| Alternativa | Descripción Técnica | Ventajas | Inconvenientes / Coste |
| :--- | :--- | :--- | :--- |
| **A. Ventana Condicional / Poda de Competidores (KISS & DRY)** | Mantener ventana $\pm 1$, pero podar la frase vecina si contiene menciones a otros bancos de `ALIAS_BANCOS` o caídas porcentuales ajenas. | **Resuelve el 100% del contagio sin añadir modelos.** Máxima pureza de código y bajo consumo de tokens. | Requiere mantener actualizada la lista de entidades excluyentes. |
| **B. Inferencia Bi-Focal con Ensamble** | Realizar dos pasadas ultra-cortas por noticia: `Score_Foco` (Ventana 0) y `Score_Contexto` (Ventana $\pm 1$). Si hay discrepancia radical, priorizar el foco. | Gran robustez matemática; detecta divergencias en tiempo real. | Duplica el número de inferencias en CPU (aunque con longitudes de <120 tokens el impacto es muy bajo). |
| **C. Modelos Orientados a Aspecto (ABSA)** | Sustituir o complementar FinBERT por un modelo de *Aspect-Based Sentiment Analysis* o inyectar etiquetas de marcado de entidad (`[E] Citi [/E]`). | Arquitectura estándar en la literatura académica reciente de NLP. | Mayor complejidad de dependencias y posible pérdida de velocidad de inferencia. |

### Conclusión y Recomendación Preliminar
La **Alternativa A (Ventana Condicional con Poda de Competidores)** es la opción recomendada: eleva la exactitud al **66.7%**, dispara el Recall Positivo al **75.0%**, conserva el **83.3%** en detección de riesgos negativos y reduce las inversiones graves de 6 a 1, respetando los principios KISS y DRY de la arquitectura.